# Export MongoDB collections to ParquetWrites `companies` and `financial_data` to Parquet under `data/parquet/`, foruse as one of four data sources in the engine benchmark.The export reads from MongoDB rather than from `enheter_alle.json`, because`financial_data` exists only in MongoDB and reading both from one sourceguarantees the variants see identical content.**Both collections are now exported at full width.** The earlier version wrotefourteen scalar columns and excluded the nested `data` statement blob, whichmeant Parquet's column-pruning advantage was banked at export time rather thanmeasured at query time — a limitation the previous notebook acknowledged. Withthe full schema, pruning is exercised where it belongs, inside the query, andthe benchmark can compare narrow and wide workloads on the same files.The only fields omitted are `links`, `organisasjonsform.links` and`foretaksformIHjemlandet.links`, all three empty in 1,171,373 of 1,171,373records. See `schemas.py` for the derivation.**Pause Dropbox sync before running.**

In [1]:
import json
import os

from pyspark.sql import SparkSession

from schemas import COMPANIES_SCHEMA, FINANCIAL_SCHEMA, EXPECTED_ROWS

MONGO_DB = "companiesdb"
DATA_DIR = "/home/jovyan/data"
PARQUET_DIR = os.path.join(DATA_DIR, "parquet")
NDJSON_DIR = os.path.join(DATA_DIR, "ndjson")
RAW_COMPANIES = os.path.join(DATA_DIR, "enheter_alle.json")

# Driver memory, thread count, the Mongo connector package and the connection
# URI all come from jupyter/spark-defaults.conf, which is baked into the image.
# Nothing about the JVM is configured here, so the notebook cannot drift from
# the environment a grader gets. Change the config file and rebuild instead.
spark = SparkSession.builder.appName("group13_parquet_export").getOrCreate()

_conf = spark.sparkContext.getConf()
MONGO_URI = _conf.get("spark.mongodb.read.connection.uri")
CONNECTOR = _conf.get("spark.jars.packages")

print("Spark        ", spark.version)
print("master       ", spark.sparkContext.master)
print("driver heap  %.1f GB" % (spark._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3))
print("connector    ", CONNECTOR)
print("mongo uri    ", MONGO_URI)
print("host cores   ", os.cpu_count())

Spark         4.2.0
master        local[4]
driver heap  8.0 GB
connector     org.mongodb.spark:mongo-spark-connector_2.13:11.1.0
mongo uri     mongodb://mongodb:27017
host cores    12


## Change detectionA full re-export takes minutes, so it is skipped when the source is unchanged.When the signature differs the whole collection is rewritten. At this size,merge logic would add failure modes to save a few minutes.Note that widening the schema does not change the signature, so the first runafter this change needs `FORCE_REFRESH = True` to discard the narrow export.

In [2]:
from datetime import datetime, timezone

from pymongo import MongoClient

METADATA_PATH = os.path.join(PARQUET_DIR, "_export_metadata.json")

# True for the first run after the schema was widened; the signature alone
# cannot detect a schema change.
FORCE_REFRESH = False

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]


def source_signature():
    """Cheap fingerprint of both collections, used to decide whether to re-export."""
    newest = db.financial_data.find_one(sort=[("fetched_at", -1)],
                                        projection={"fetched_at": 1})
    return {
        "companies": {"count": db.companies.count_documents({})},
        "financial_data": {
            "count": db.financial_data.count_documents({}),
            "max_fetched_at": newest["fetched_at"].isoformat() if newest and newest.get("fetched_at") else None,
        },
    }


previous = {}
if os.path.exists(METADATA_PATH):
    with open(METADATA_PATH) as fh:
        previous = json.load(fh).get("signature", {})

current = source_signature()
stale = {name: (FORCE_REFRESH or current[name] != previous.get(name)) for name in current}

print(json.dumps(current, indent=2))
print()
for name, needed in stale.items():
    print("%-16s %s" % (name, "export needed" if needed else "unchanged, skipping"))

{
  "companies": {
    "count": 1171373
  },
  "financial_data": {
    "count": 1170290,
    "max_fetched_at": "2026-08-29T06:58:51.379000"
  }
}

companies        export needed
financial_data   export needed


In [3]:
def export(collection, schema):
    df = (
        spark.read.format("mongodb")
        .option("database", MONGO_DB)
        .option("collection", collection)
        .schema(schema)
        .load()
    )
    target = os.path.join(PARQUET_DIR, collection)
    df.write.mode("overwrite").parquet(target)
    # Count from the written files rather than the DataFrame, which would
    # otherwise re-read the whole collection from MongoDB a second time.
    return spark.read.parquet(target).count()


os.makedirs(PARQUET_DIR, exist_ok=True)
written = {}

for name, schema in [("companies", COMPANIES_SCHEMA), ("financial_data", FINANCIAL_SCHEMA)]:
    if not stale[name]:
        print("Skipping %s (unchanged)" % name)
        continue
    print("Exporting %s ..." % name)
    written[name] = export(name, schema)
    print("  wrote %d rows" % written[name])

if written:
    with open(METADATA_PATH, "w") as fh:
        json.dump({"signature": current,
                   "exported_at": datetime.now(timezone.utc).isoformat(),
                   "rows_written": written}, fh, indent=2)
    print("\nMetadata updated.")

Exporting companies ...
  wrote 1171373 rows
Exporting financial_data ...
  wrote 1170290 rows

Metadata updated.


## VerificationRow counts must match MongoDB exactly, and the widened columns must bepopulated at the frequencies the profiling pass recorded. A count match alonewould not catch a nested field that read as null throughout.

In [4]:
from pyspark.sql import functions as F

for name in ["companies", "financial_data"]:
    path = os.path.join(PARQUET_DIR, name)
    parquet_rows = spark.read.parquet(path).count()
    mongo_rows = db[name].count_documents({})
    print("%-16s parquet=%d  mongo=%d  %s"
          % (name, parquet_rows, mongo_rows,
             "OK" if parquet_rows == mongo_rows else "MISMATCH"))
    assert parquet_rows == EXPECTED_ROWS[name], "unexpected row count for %s" % name

# Expected non-null counts come from the full profiling pass, not from a sample.
comp = spark.read.parquet(os.path.join(PARQUET_DIR, "companies"))
fin = spark.read.parquet(os.path.join(PARQUET_DIR, "financial_data"))

expect = [
    (comp, "naeringskode1.kode", 1135646),
    (comp, "forretningsadresse.kommunenummer", 1108985),
    (comp, "kapital.belop", 437842),
    (comp, "naeringskode3.kode", 1576),
    (fin, "data", 444644),
    (fin, "data.resultatregnskapResultat.totalresultat", None),
]

print("\n%-52s %10s %10s" % ("column", "non-null", "expected"))
print("-" * 76)
for df, col, exp in expect:
    if exp is None:
        continue
    n = df.filter(F.col(col).isNotNull()).count()
    print("%-52s %10d %10d %s" % (col, n, exp, "OK" if n == exp else "MISMATCH"))

# Nested inside the array, so addressed by element rather than by path.
n = fin.filter(F.col("data")[0]["resultatregnskapResultat"]["totalresultat"].isNotNull()).count()
print("%-52s %10d %10d %s" % ("data[0]...totalresultat", n, 241560, "OK" if n == 241560 else "MISMATCH"))

size = sum(os.path.getsize(os.path.join(d, f))
           for d, _, files in os.walk(PARQUET_DIR) for f in files)
print("\nParquet total on disk: %.2f GB" % (size / 1024**3))

companies        parquet=1171373  mongo=1171373  OK
financial_data   parquet=1170290  mongo=1170290  OK

column                                                 non-null   expected
----------------------------------------------------------------------------
naeringskode1.kode                                      1135646    1135646 OK
forretningsadresse.kommunenummer                        1108985    1108985 OK
kapital.belop                                            437842     437842 OK
naeringskode3.kode                                         1576       1576 OK
data                                                     444644     444644 OK
data[0]...totalresultat                                  241560     241560 OK

Parquet total on disk: 0.26 GB
